### EXPLORACIÓN INICIAL DE LOS DATOS 

### Carga de los datos 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../Base_de_datos.csv")
print(df.shape)
df.head()

(10763, 23)


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,4,2025-01-07 14:40:36,1852560.0,12,32,Empleado,3500000,1000000,128650,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,916148.0,Creciente,1
1,4,2025-01-09 11:18:48,3181080.0,6,34,Empleado,5000000,2000000,441817,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,4473774.0,Creciente,1
2,9,2024-12-26 18:52:57,670200.0,5,34,Independiente,5000000,2000000,108632,95.227787,...,0.0,274561.0,274561.0,NaN,2,0,1,NaN,NaN,1
3,9,2024-12-04 14:20:50,506806.8,2,25,Independiente,3000000,900000,199684,95.227787,...,0.0,27564.0,27564.0,NaN,1,0,6,NaN,NaN,1
4,4,2025-04-30 18:41:57,999780.0,10,26,Empleado,2000000,600000,92509,95.227787,...,0.0,0.0,NaN,NaN,0,0,0,61000.0,Creciente,1


### Descripción general y caracterización de los datos

El dataset tiene 10.763 registros y 23 columnas. A continuación se clasifica cada variable:

| Variable | Tipo | Naturaleza | Escala | Observaciones |
|---|---|---|---|---|
| tipo_credito | Categórica | Cualitativo | Nominal | Politómica  |
| fecha_prestamo | Numérica | Cuantitativo | Intervalo | Fecha; continua en el tiempo |
| capital_prestado | Numérica | Cuantitativo | Razón | Continua |
| plazo_meses | Numérica | Cuantitativo | Razón | Discreta |
| edad_cliente | Numérica | Cuantitativo | Razón | Discreta |
| tipo_laboral | Categórica | Cualitativo | Nominal | Dicotómica (Empleado / Independiente) |
| salario_cliente | Numérica | Cuantitativo | Razón | Continua |
| total_otros_prestamos | Numérica | Cuantitativo | Razón | Continua |
| cuota_pactada | Numérica | Cuantitativo | Razón | Continua |
| puntaje | Numérica | Cuantitativo | Intervalo | Continua; tiene valores negativos, el cero no es ausencia |
| puntaje_datacredito | Numérica | Cuantitativo | Intervalo | Discreta |
| cant_creditosvigentes | Numérica | Cuantitativo | Razón | Discreta |
| huella_consulta | Numérica | Cuantitativo | Razón | Discreta |
| saldo_mora | Numérica | Cuantitativo | Razón | Continua |
| saldo_total | Numérica | Cuantitativo | Razón | Continua |
| saldo_principal | Numérica | Cuantitativo | Razón | Continua |
| saldo_mora_codeudor | Numérica | Cuantitativo | Razón | Continua |
| creditos_sectorFinanciero | Numérica | Cuantitativo | Razón | Discreta |
| creditos_sectorCooperativo | Numérica | Cuantitativo | Razón | Discreta |
| creditos_sectorReal | Numérica | Cuantitativo | Razón | Discreta |
| promedio_ingresos_datacredito | Numérica | Cuantitativo | Razón | Continua |
| tendencia_ingresos | Categórica | Cualitativo | Ordinal | Politómica (Decreciente < Estable < Creciente) |
| Pago_atiempo | Categórica | Cualitativo | Nominal | Dicotómica; variable objetivo (0 / 1) |

### Revisión de los nulos

Conteo de nulos por columna

In [2]:
nulos = df.isnull().sum()
tabla_nulos = pd.DataFrame({
    "nulos": nulos,
    "porcentaje": (nulos / len(df) * 100).round(2)
})
tabla_nulos[tabla_nulos["nulos"] > 0].sort_values("porcentaje", ascending=False)

,nulos,porcentaje
tendencia_ingresos,2932,27.24
promedio_ingresos_datacredito,2930,27.22
saldo_mora_codeudor,590,5.48
saldo_principal,405,3.76
saldo_mora,156,1.45
saldo_total,156,1.45
puntaje_datacredito,6,0.06


Revisión de Categorías de tipo_laboral y tendencia_ingresos

In [3]:
for col in ["tipo_laboral", "tendencia_ingresos"]:
    print(col)
    print(df[col].value_counts(dropna=False))
    print()

tipo_laboral
tipo_laboral
Empleado         6754
Independiente    4009
Name: count, dtype: int64

tendencia_ingresos
tendencia_ingresos
Creciente      5294
NaN            2932
Decreciente    1291
Estable        1188
0                 7
8315              6
1000000           4
9147              2
158042            1
3978              1
168750            1
-28589            1
-566272           1
24702             1
31837             1
122727            1
417087            1
9090              1
173031            1
-70715            1
-435177           1
-702927           1
-4105             1
54683             1
22832             1
209090            1
5697              1
-288              1
-164315           1
2029000           1
17181             1
15245             1
82657             1
52862             1
1817052           1
75761             1
146918            1
1123000           1
15090             1
4250635           1
22363             1
-101368           1
86286             1
65988

Corrección de tendencia_ingresos: valores inválidos a NaN

In [4]:
validos = ["Creciente", "Decreciente", "Estable"]
invalidos = df["tendencia_ingresos"].notna() & ~df["tendencia_ingresos"].isin(validos)

print("Valores inválidos encontrados:", invalidos.sum())

df.loc[invalidos, "tendencia_ingresos"] = np.nan

df["tendencia_ingresos"].value_counts(dropna=False)

Valores inválidos encontrados: 58


tendencia_ingresos
Creciente      5294
NaN            2990
Decreciente    1291
Estable        1188
Name: count, dtype: int64

DETECCIÓN DE VALORES SOSPECHOSOS EN VARIABLES NUMÉRICAS

In [5]:
num = df.select_dtypes(include="number")

revision = pd.DataFrame({
    "min": num.min(),
    "max": num.max(),
    "ceros": (num == 0).sum(),
    "negativos": (num < 0).sum(),
    "valor_mas_repetido": num.mode().iloc[0],
    "veces_repetido": num.apply(lambda s: s.value_counts().iloc[0]),
})
revision

,min,max,ceros,negativos,valor_mas_repetido,veces_repetido
tipo_credito,4.00000,6.800000e+01,0,0,4.000000e+00,7747
capital_prestado,360000.00000,4.144415e+07,0,0,1.200000e+06,129
plazo_meses,2.00000,9.000000e+01,0,0,6.000000e+00,3326
edad_cliente,19.00000,1.230000e+02,0,0,4.000000e+01,346
salario_cliente,0.00000,2.200000e+10,24,0,3.000000e+06,829
total_otros_prestamos,0.00000,6.787675e+09,445,0,2.000000e+06,1109
cuota_pactada,23944.00000,3.816752e+06,0,0,2.059900e+05,13
puntaje,-38.00999,9.522779e+01,0,135,9.522779e+01,9407
puntaje_datacredito,-7.00000,9.990000e+02,145,1,0.000000e+00,145
cant_creditosvigentes,0.00000,6.200000e+01,405,0,4.000000e+00,1396


Conteo de filas con valores sospechosos

In [6]:
sospechosos = {
    "edad_cliente > 100": (df["edad_cliente"] > 100).sum(),
    "salario_cliente == 0": (df["salario_cliente"] == 0).sum(),
    "salario_cliente > 100 millones": (df["salario_cliente"] > 100_000_000).sum(),
    "puntaje < 0": (df["puntaje"] < 0).sum(),
    "puntaje_datacredito <= 0": (df["puntaje_datacredito"] <= 0).sum(),
    "puntaje_datacredito == 999": (df["puntaje_datacredito"] == 999).sum(),
}
pd.Series(sospechosos, name="filas")

edad_cliente > 100                150
salario_cliente == 0               24
salario_cliente > 100 millones     74
puntaje < 0                       135
puntaje_datacredito <= 0          146
puntaje_datacredito == 999          1
Name: filas, dtype: int64

Inspección de los valores sospechosos

In [7]:
print("Edades > 100:")
print(df.loc[df["edad_cliente"] > 100, "edad_cliente"].value_counts().head(10))

print("\nSalarios > 100 millones (los 10 más grandes):")
print(df.loc[df["salario_cliente"] > 100_000_000, "salario_cliente"].sort_values(ascending=False).head(10))

print("\npuntaje_datacredito <= 0:")
print(df.loc[df["puntaje_datacredito"] <= 0, "puntaje_datacredito"].value_counts())

print("\npuntaje < 0 (resumen):")
print(df.loc[df["puntaje"] < 0, "puntaje"].describe())

Edades > 100:
edad_cliente
122    144
121      4
123      2
Name: count, dtype: int64

Salarios > 100 millones (los 10 más grandes):
5515     22000000000
2737     16991100000
10476    13258352050
10505    12450806583
10503     8252584000
10532     7170914725
10515     4857353453
10578     4663274549
10572     4067140000
10520     3979311000
Name: salario_cliente, dtype: int64

puntaje_datacredito <= 0:
puntaje_datacredito
 0.0    145
-7.0      1
Name: count, dtype: int64

puntaje < 0 (resumen):
count    135.000000
mean     -10.975885
std        6.591633
min      -38.009990
25%      -16.250449
50%      -11.219562
75%       -5.102625
max       -0.367477
Name: puntaje, dtype: float64


Corrección de edad_cliente y puntaje_datacredito: códigos a NaN

In [8]:
df.loc[df["edad_cliente"] > 100, "edad_cliente"] = np.nan
df.loc[df["puntaje_datacredito"].isin([0, -7, 999]), "puntaje_datacredito"] = np.nan

df[["edad_cliente", "puntaje_datacredito"]].isnull().sum()

edad_cliente           150
puntaje_datacredito    153
dtype: int64

Revisión de ceros y valores bajos pendientes

In [9]:
print("puntaje_datacredito menor a 400:")
print(df.loc[df["puntaje_datacredito"] < 400, "puntaje_datacredito"].sort_values())

print("\nsalario_cliente == 0:")
print(df.loc[df["salario_cliente"] == 0, ["tipo_laboral", "promedio_ingresos_datacredito", "Pago_atiempo"]])

print("\npromedio_ingresos_datacredito == 0:")
print(df.loc[df["promedio_ingresos_datacredito"] == 0, ["salario_cliente", "tipo_laboral", "Pago_atiempo"]])

puntaje_datacredito menor a 400:
6798      7.0
4258    287.0
7310    342.0
9358    359.0
6966    380.0
915     390.0
7337    395.0
8827    398.0
Name: puntaje_datacredito, dtype: float64

salario_cliente == 0:
        tipo_laboral  promedio_ingresos_datacredito  Pago_atiempo
125         Empleado                      1350222.0             1
281    Independiente                      3000000.0             1
380    Independiente                            NaN             1
445         Empleado                      1760890.0             1
446    Independiente                       905161.0             1
1008   Independiente                       885813.0             1
1340        Empleado                       923771.0             1
1415   Independiente                       615779.0             1
2735        Empleado                       905965.0             1
3279   Independiente                            NaN             1
3608   Independiente                       961885.0             

Corrección de puntaje_datacredito, salario_cliente y promedio_ingresos: 7 y ceros a NaN

In [10]:
df.loc[df["puntaje_datacredito"] == 7, "puntaje_datacredito"] = np.nan
df.loc[df["salario_cliente"] == 0, "salario_cliente"] = np.nan
df.loc[df["promedio_ingresos_datacredito"] == 0, "promedio_ingresos_datacredito"] = np.nan

df[["puntaje_datacredito", "salario_cliente", "promedio_ingresos_datacredito"]].isnull().sum()

puntaje_datacredito               154
salario_cliente                    24
promedio_ingresos_datacredito    2937
dtype: int64

Relación entre los nulos de ingresos

In [11]:
sin_tendencia = df["tendencia_ingresos"].isnull()
sin_promedio = df["promedio_ingresos_datacredito"].isnull()

pd.crosstab(sin_tendencia, sin_promedio,
            rownames=["tendencia_ingresos es nulo"],
            colnames=["promedio_ingresos es nulo"])

promedio_ingresos es nulo,False,True
tendencia_ingresos es nulo,,
False,7766,7
True,60,2930


Nulos de las variables de saldos

In [12]:
saldos = ["saldo_mora", "saldo_total", "saldo_principal", "saldo_mora_codeudor"]
print(df[saldos].isnull().sum())
print()
print("Nulos en saldo_mora y saldo_total a la vez:",
      (df["saldo_mora"].isnull() & df["saldo_total"].isnull()).sum())
print("Nulos en saldo_principal con cant_creditosvigentes == 0:",
      (df["saldo_principal"].isnull() & (df["cant_creditosvigentes"] == 0)).sum())

saldo_mora             156
saldo_total            156
saldo_principal        405
saldo_mora_codeudor    590
dtype: int64

Nulos en saldo_mora y saldo_total a la vez: 156
Nulos en saldo_principal con cant_creditosvigentes == 0: 405


Origen de los nulos de saldo_mora y saldo_mora_codeudor

In [13]:
sin_mora = df["saldo_mora"].isnull()
print("Entre los 156 con saldo_mora nulo:")
print("  con cant_creditosvigentes == 0:", (sin_mora & (df["cant_creditosvigentes"] == 0)).sum())
print("  con saldo_principal nulo:", (sin_mora & df["saldo_principal"].isnull()).sum())

sin_cod = df["saldo_mora_codeudor"].isnull()
print()
print("Entre los 590 con saldo_mora_codeudor nulo:")
print("  con cant_creditosvigentes == 0:", (sin_cod & (df["cant_creditosvigentes"] == 0)).sum())
print("  con saldo_mora nulo:", (sin_cod & sin_mora).sum())
print("  con saldo_principal nulo:", (sin_cod & df["saldo_principal"].isnull()).sum())

Entre los 156 con saldo_mora nulo:
  con cant_creditosvigentes == 0: 156
  con saldo_principal nulo: 156

Entre los 590 con saldo_mora_codeudor nulo:
  con cant_creditosvigentes == 0: 405
  con saldo_mora nulo: 156
  con saldo_principal nulo: 405


Nulos de saldo_mora_codeudor en clientes con créditos vigentes

In [14]:
restantes = df["saldo_mora_codeudor"].isnull() & (df["cant_creditosvigentes"] > 0)
print("Nulos de saldo_mora_codeudor con créditos vigentes:", restantes.sum())
print()
print(df.loc[restantes, "tipo_credito"].value_counts())
print()
print(df.loc[restantes, "tipo_laboral"].value_counts())
print()
print("Promedio de Pago_atiempo en esos nulos:", df.loc[restantes, "Pago_atiempo"].mean().round(3))
print("Promedio de Pago_atiempo en toda la base:", df["Pago_atiempo"].mean().round(3))

Nulos de saldo_mora_codeudor con créditos vigentes: 185

tipo_credito
9    169
4     16
Name: count, dtype: int64

tipo_laboral
Empleado         95
Independiente    90
Name: count, dtype: int64

Promedio de Pago_atiempo en esos nulos: 0.941
Promedio de Pago_atiempo en toda la base: 0.953


Revisión de puntaje: valor tope

In [15]:
tope = df["puntaje"].max()
es_tope = df["puntaje"] == tope

print("Valor máximo:", tope)
print("Filas con ese valor:", es_tope.sum(), f"({es_tope.mean()*100:.1f}%)")
print()
print("Resumen de las filas que NO son el tope:")
print(df.loc[~es_tope, "puntaje"].describe().round(2))
print()
print("Promedio de Pago_atiempo con el tope:", df.loc[es_tope, "Pago_atiempo"].mean().round(3))
print("Promedio de Pago_atiempo sin el tope:", df.loc[~es_tope, "Pago_atiempo"].mean().round(3))

Valor máximo: 95.227787
Filas con ese valor: 9407 (87.4%)

Resumen de las filas que NO son el tope:
count    1356.00
mean       63.02
std        35.30
min       -38.01
25%        44.36
50%        84.73
75%        89.98
max        92.76
Name: puntaje, dtype: float64

Promedio de Pago_atiempo con el tope: 1.0
Promedio de Pago_atiempo sin el tope: 0.623


Revisión de total_otros_prestamos

In [16]:
col = df["total_otros_prestamos"]
print("Valores distintos:", col.nunique())
print()
print(col.describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(0))
print()
print("Ceros:", (col == 0).sum())
print("Ceros con cant_creditosvigentes == 0:", ((col == 0) & (df["cant_creditosvigentes"] == 0)).sum())
print("Ceros con saldo_total nulo:", ((col == 0) & df["saldo_total"].isnull()).sum())
print("Promedio de Pago_atiempo en los ceros:", df.loc[col == 0, "Pago_atiempo"].mean().round(3))

Valores distintos: 1538

count    1.076300e+04
mean     6.238870e+06
std      1.184183e+08
min      0.000000e+00
1%       0.000000e+00
25%      5.000000e+05
50%      1.000000e+06
75%      2.000000e+06
95%      4.798900e+06
99%      1.794606e+07
max      6.787675e+09
Name: total_otros_prestamos, dtype: float64

Ceros: 445
Ceros con cant_creditosvigentes == 0: 60
Ceros con saldo_total nulo: 12
Promedio de Pago_atiempo en los ceros: 0.908


Revisión de cuota, conteos y validación de sectores

In [17]:
cols = ["cuota_pactada", "cant_creditosvigentes", "huella_consulta",
        "creditos_sectorFinanciero", "creditos_sectorCooperativo", "creditos_sectorReal"]
print(df[cols].describe(percentiles=[.01, .5, .99]).round(1).T)

sectores = ["creditos_sectorFinanciero", "creditos_sectorCooperativo", "creditos_sectorReal"]
suma = df[sectores].sum(axis=1)
print()
print("Suma de sectores = cant_creditosvigentes:", (suma == df["cant_creditosvigentes"]).sum())
print("Suma de sectores > cant_creditosvigentes:", (suma > df["cant_creditosvigentes"]).sum())
print("Suma de sectores < cant_creditosvigentes:", (suma < df["cant_creditosvigentes"]).sum())

                              count      mean       std      min       1%  \
cuota_pactada               10763.0  243617.4  210493.7  23944.0  51439.9   
cant_creditosvigentes       10763.0       5.7       4.0      0.0      0.0   
huella_consulta             10763.0       4.2       3.1      0.0      0.0   
creditos_sectorFinanciero   10763.0       2.8       2.7      0.0      0.0   
creditos_sectorCooperativo  10763.0       0.3       0.7      0.0      0.0   
creditos_sectorReal         10763.0       1.3       1.8      0.0      0.0   

                                 50%        99%        max  
cuota_pactada               182863.0  1114004.4  3816752.0  
cant_creditosvigentes            5.0       19.0       62.0  
huella_consulta                  4.0       15.0       29.0  
creditos_sectorFinanciero        2.0       12.0       51.0  
creditos_sectorCooperativo       0.0        3.0       13.0  
creditos_sectorReal              1.0        8.0       25.0  

Suma de sectores = cant_creditos

## Resumen: unificación de valores nulos

Todos los datos faltantes quedaron representados como `NaN`. Las decisiones se tomaron por los patrones de los datos, ya que no hay diccionario de datos.

**Valores convertidos a `NaN`:**

| Variable | Valores reemplazados | Filas |
|---|---|---|
| `tendencia_ingresos` | Números en una columna que solo admite Creciente, Decreciente o Estable | 58 |
| `edad_cliente` | Edades de 121 a 123 años (código de "sin dato") | 150 |
| `puntaje_datacredito` | 0, -7, 999 y 7 (no son puntajes reales) | 148 |
| `salario_cliente` | 0 (clientes con ingreso reportado en la central) | 24 |
| `promedio_ingresos_datacredito` | 0 (clientes con salario declarado) | 7 |

**Se conservan:** los ceros de saldos, créditos y `total_otros_prestamos` (datos reales) y los valores extremos de `salario_cliente` y `puntaje`, que se tratarán como outliers.

**Nulos finales:**

| Variable | Nulos | % |
|---|---|---|
| tendencia_ingresos | 2.990 | 27.78 |
| promedio_ingresos_datacredito | 2.937 | 27.29 |
| saldo_mora_codeudor | 590 | 5.48 |
| saldo_principal | 405 | 3.76 |
| saldo_mora | 156 | 1.45 |
| saldo_total | 156 | 1.45 |
| puntaje_datacredito | 154 | 1.43 |
| edad_cliente | 150 | 1.39 |
| salario_cliente | 24 | 0.22 |

Los nulos de `saldo_principal`, `saldo_mora` y `saldo_total` corresponden a clientes sin créditos vigentes, y los de `tendencia_ingresos` y `promedio_ingresos_datacredito` faltan casi siempre a la vez (clientes sin historial de ingresos). La imputación se hará en `ft_engineering.py`.

### 1.6 Eliminación de variables irrelevantes

#### Revisión de variables constantes, identificadores y con muchos nulos

In [18]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "distintos": df.nunique(),
    "pct_nulos": (df.isnull().mean() * 100).round(1),
    "pct_valor_mas_frecuente": (
        df.apply(lambda s: s.value_counts(normalize=True).iloc[0]) * 100
    ).round(1),
})

print("Filas duplicadas:", df.duplicated().sum())
resumen.sort_values("pct_valor_mas_frecuente", ascending=False)

Filas duplicadas: 0


,tipo,distintos,pct_nulos,pct_valor_mas_frecuente
saldo_mora_codeudor,float64,4,5.5,100.0
saldo_mora,float64,55,1.4,99.5
Pago_atiempo,int64,2,0.0,95.3
puntaje,float64,248,0.0,87.4
creditos_sectorCooperativo,int64,11,0.0,82.4
tipo_credito,int64,6,0.0,72.0
tendencia_ingresos,str,3,27.8,68.1
tipo_laboral,str,2,0.0,62.8
creditos_sectorReal,int64,24,0.0,41.1
plazo_meses,int64,18,0.0,30.9


In [19]:
for col in ["saldo_mora_codeudor", "saldo_mora"]:
    con_valor = df[col] > 0
    sin_valor = df[col] == 0
    print(col)
    print("  Filas con valor mayor a 0:", con_valor.sum())
    print("  Pago a tiempo en esas filas:", df.loc[con_valor, "Pago_atiempo"].mean().round(3))
    print("  Pago a tiempo en las filas con 0:", df.loc[sin_valor, "Pago_atiempo"].mean().round(3))
    print()

saldo_mora_codeudor
  Filas con valor mayor a 0: 3
  Pago a tiempo en esas filas: 1.0
  Pago a tiempo en las filas con 0: 0.954

saldo_mora
  Filas con valor mayor a 0: 55
  Pago a tiempo en esas filas: 0.636
  Pago a tiempo en las filas con 0: 0.955



In [21]:
dif = (df["saldo_total"] - df["saldo_principal"] - df["saldo_mora"]).abs()
print("Filas donde saldo_total = saldo_principal + saldo_mora:", (dif < 1).sum(), "de", dif.notna().sum())
print()
print(df[["saldo_total", "saldo_principal", "saldo_mora"]].corr().round(3))

Filas donde saldo_total = saldo_principal + saldo_mora: 8609 de 10358

                 saldo_total  saldo_principal  saldo_mora
saldo_total            1.000            0.735       0.002
saldo_principal        0.735            1.000       0.005
saldo_mora             0.002            0.005       1.000


#### Eliminación de saldo_mora_codeudor

Se elimina `saldo_mora_codeudor` porque solo 3 de sus 10.173 registros con dato tienen un valor distinto de 0, además de un 5.5% de nulos. Con tan pocos casos no aporta información para distinguir a los clientes. Las demás variables se conservan: ninguna es identificador, constante, duplicada ni tiene demasiados nulos. `saldo_mora`, aunque casi siempre vale 0, se conserva porque los 55 clientes con mora pagan a tiempo el 63.6% frente al 95.5% del resto.

In [22]:
df = df.drop(columns=["saldo_mora_codeudor"])
print("Columnas restantes:", df.shape[1])

Columnas restantes: 22


### 1.7 Conversión de tipos de datos

#### Conversión de variables a su tipo correcto

- `fecha_prestamo`: de texto a fecha (`datetime`).
- `tipo_credito`: de entero a categórica, porque son códigos de categoría y no cantidades.
- `tipo_laboral`: de texto a categórica.
- `tendencia_ingresos`: categórica ordinal (Decreciente < Estable < Creciente).
- `Pago_atiempo`: de entero a categórica binaria (variable objetivo).
- Las demás variables se mantienen numéricas.

In [23]:
df["fecha_prestamo"] = pd.to_datetime(df["fecha_prestamo"])

df["tipo_credito"] = df["tipo_credito"].astype("category")
df["tipo_laboral"] = df["tipo_laboral"].astype("category")
df["Pago_atiempo"] = df["Pago_atiempo"].astype("category")

orden = ["Decreciente", "Estable", "Creciente"]
df["tendencia_ingresos"] = pd.Categorical(df["tendencia_ingresos"], categories=orden, ordered=True)

print("Nulos en tendencia_ingresos:", df["tendencia_ingresos"].isnull().sum())
print("Fechas desde", df["fecha_prestamo"].min(), "hasta", df["fecha_prestamo"].max())
print()
df.dtypes

Nulos en tendencia_ingresos: 2990
Fechas desde 2024-11-26 09:17:04 hasta 2026-04-26 18:43:52



tipo_credito                           category
fecha_prestamo                   datetime64[us]
capital_prestado                        float64
plazo_meses                               int64
edad_cliente                            float64
tipo_laboral                           category
salario_cliente                         float64
total_otros_prestamos                     int64
cuota_pactada                             int64
puntaje                                 float64
puntaje_datacredito                     float64
cant_creditosvigentes                     int64
huella_consulta                           int64
saldo_mora                              float64
saldo_total                             float64
saldo_principal                         float64
creditos_sectorFinanciero                 int64
creditos_sectorCooperativo                int64
creditos_sectorReal                       int64
promedio_ingresos_datacredito           float64
tendencia_ingresos                     c